In [1]:
import sys, json, warnings
sys.path.insert(0, "..")            # ml-service root, so `training` and `app` import
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from training import config as C
sns.set_theme(style="whitegrid")
IMG = C.ROOT_DIR / "docs" / "images"; IMG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='1500MB'"); con.execute("SET threads=2")
G = str(C.FLIGHTS_GLOB).replace("\\", "/")
def q(sql): return con.execute(sql.replace("{G}", G)).df()
def save(name):
    plt.tight_layout(); plt.savefig(IMG / f"{name}.png", dpi=110); plt.show()
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from training.feature_engineering import load_xy
from training.evaluate import best_f1_threshold

# 04 - Class imbalance: weights vs SMOTE (comparison only)
The CANCEL task is the imbalanced one (about 1.6% positives in train). Production uses class weights (Decision 9). SMOTE is tried here on a sample only to justify that choice.

In [2]:
Xtr, ytr = load_xy("train", "cancel", sample=0.12); Xva, yva = load_xy("valid", "cancel", sample=0.5)
P = dict(n_estimators=250, learning_rate=0.1, max_depth=6, tree_method="hist", n_jobs=3, eval_metric="aucpr", early_stopping_rounds=20, random_state=42)
def run(name, X, y, spw=1.0):
    m = xgb.XGBClassifier(scale_pos_weight=spw, **P).fit(X, y, eval_set=[(Xva, yva)], verbose=False)
    p = m.predict_proba(Xva)[:, 1]; t = best_f1_threshold(yva, p)
    return {"strategy": name, "rows": len(y), "roc_auc": roc_auc_score(yva, p), "pr_auc": average_precision_score(yva, p), "f1@best": f1_score(yva, p >= t)}
res = [run("none", Xtr, ytr), run("class weights", Xtr, ytr, spw=(ytr == 0).sum() / (ytr == 1).sum())]
Xs, ys = SMOTE(sampling_strategy=0.2, random_state=42, k_neighbors=5).fit_resample(Xtr, ytr)
res.append(run("SMOTE 20%", Xs, ys))
pd.DataFrame(res).round(4)

,strategy,rows,roc_auc,pr_auc,f1@best
0,none,543827,0.8403,0.0804,0.1575
1,class weights,543827,0.8418,0.0923,0.1559
2,SMOTE 20%,641824,0.8447,0.0965,0.1667


Finding: the three strategies are close. Class weights lift PR-AUC over no handling (0.092 vs 0.080); SMOTE 20% is a little higher again on PR-AUC / F1 (0.097 / 0.167 vs 0.092 / 0.156) but that is a marginal gain on one validation month, it adds 18% more training rows, and synthetic positives distort probabilities, which isotonic calibration would have to undo anyway. We keep class weights + isotonic calibration (Decision 9); SMOTE is reported here for comparison only.